# Walk Attention — un tutorial centrado en la teoría

Álgebras de caminos de quivers como una atención sparse multi-salto para mitigar el over-squashing en GNNs. Este único cuaderno recorre toda la idea desde cero, con figuras teóricas hechas a mano y código ejecutable de `aiq-quivers` / `networkx`. Léelo de arriba a abajo.

## Contenido
0. Parte 0 — Quivers, caminos y multiplicidad
1. Parte 1 — ¿Qué es el over-squashing?
2. Parte 2 — El álgebra de caminos: contar caminos
3. Parte 3 — Los caminos son mensajes
4. Parte 4 — El operador de walk es atención
5. Parte 5 — Juntándolo todo

# P0 — Quivers, caminos y multiplicidad

Todo en este proyecto descansa en un objeto: un **quiver** (un grafo dirigido) y los **caminos** en él. Este
cuaderno construye el vocabulario con diagramas hechos a mano, y te deja *construir los grafos tú mismo* con
la librería `aiq-quivers`.

## 1. ¿Qué es un quiver?

<img src="../figs-theory/es/T1_quiver_anatomy.svg" width="760"/>

In [ ]:
# Construye ese quiver con aiq-quivers. Las flechas son ternas (nombre, fuente, objetivo).
from aiq.quiver import Quiver
Q = Quiver([1, 2, 3], [('a', 1, 2), ('aprime', 1, 2), ('b', 2, 3), ('c', 3, 3)])
print('vértices Q0:', Q.Q0)
print('matriz de adyacencia A (A[i,j] = nº de flechas i->j):')
print(Q.adjacency_matrix().astype(int))

## 2. ¿Qué es un camino?

Un camino es un recorrido dirigido: flechas unidas cabeza-con-cola. Se leen de derecha a izquierda, como la
composición.

<img src="../figs-theory/es/E0a_what_is_a_path.svg" width="760"/>

In [ ]:
# Dejamos que aiq-quivers enumere los caminos por nosotros.
from aiq.path_algebra import PathAlgebra
linea = Quiver([1,2,3,4], [('a1',1,2),('a2',2,3),('a3',3,4)])
caminos = PathAlgebra(linea).paths_from_to(1, 4, 3)
print('caminos de longitud 3 de 1 a 4:', [str(p) for p in caminos])

## 3. Los recorridos pueden repetir — y eso es lo que contamos

<img src="../figs-theory/es/E0c_path_vs_walk.svg" width="760"/>

## 4. Multiplicidad: 1 vs 2 vs 4 caminos paralelos

¿Cuántos caminos conectan dos nodos? Ese número — la **multiplicidad** — es lo que provoca el over-squashing (P1).

<img src="../figs-theory/es/E0b_multiplicity_1_2_4.svg" width="760"/>

In [ ]:
# Velo concretamente: construye los tres grafos y cuenta los caminos fuente->objetivo.
import numpy as np
from oversquash import viz

grafos = {
    'línea (n=1)':    ([(0,1),(1,2)], 3),
    'diamante (n=2)': ([(0,1),(0,2),(1,3),(2,3)], 4),
    'tres (n=3)':     ([(0,1),(0,2),(0,3),(1,4),(2,4),(3,4)], 5),
}
for nombre, (edges, n) in grafos.items():
    Qi = Quiver(list(range(n)), [(f'e{k}', u, v) for k,(u,v) in enumerate(edges)])
    A = Qi.adjacency_matrix(); g = 2
    cuenta = int(np.linalg.matrix_power(A, g)[0, n-1])
    print(f'{nombre:16s}: {cuenta} camino(s) de longitud {g} de 0 a {n-1}')

In [ ]:
# Dibuja el diamante para verlo (networkx).
viz.draw_graph([(0,1),(0,2),(1,3),(2,3)], 4, src=0, dst=3,
               pos={0:(0,0),1:(1,1),2:(1,-1),3:(2,0)}, title='el diamante: 2 caminos paralelos 0->3')

**Siguiente (P1):** más caminos paralelos de la misma longitud significa más mensajes apretados — over-squashing.

---

# P1 — ¿Qué es el over-squashing?

Una GNN de paso de mensajes actualiza cada nodo mezclando a sus vecinos, un salto a la vez. Cuando muchos
caminos confluyen en un nodo, todos sus mensajes se aplastan en un vector de ancho fijo y se pierde señal. Lo
hacemos preciso en un grafo **cuello de botella** que puedes construir con `aiq-quivers`.

## 1. El paso de mensajes avanza un salto por capa

Para alcanzar un nodo a `g` saltos necesitas `g` capas — y todo lo del camino se vuelve a apretar en cada paso.

<img src="../figs-theory/es/E1a_message_one_hop.svg" width="760"/>

## 2. El cuello de botella y la fórmula K·M^d

<img src="../figs-theory/es/T2_bottleneck_formula.svg" width="760"/>

In [ ]:
# Construye el cuello de botella con el helper de aiq-quivers y míralo.
import torch
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash import viz
K, M = 4, 3
data = make_bottleneck_graph(K, M, depth=2, generator=torch.Generator().manual_seed(0))
viz.draw_bottleneck(data, K, M, title='K fuentes -> d capas de M -> objetivo')

## 3. Un vector de ancho fijo se desborda

El conteo de recorridos hacia el objetivo crece como `K·M^d`, pero el vector del objetivo no cambia de tamaño.

<img src="../figs-theory/es/E1b_vector_overflow.svg" width="760"/>

In [ ]:
# Cuenta los mensajes forzados al objetivo, para profundidad creciente (vía operadores de walk).
from oversquash.ideal_bridge import walk_operators
for d in [1, 2, 3, 4]:
    dd = make_bottleneck_graph(K, M, d, torch.Generator().manual_seed(0))
    raw, _ = walk_operators(dd.edge_index.numpy(), dd.num_nodes, max_length=d+1)
    t = int(dd.root_mask.nonzero()[0])
    print(f'profundidad {d}: {int(raw[d][:, t].sum()):4d} mensajes apretados en un vector  (~ K*M^d)')

## 4. El conteo de recorridos explota con la distancia

<img src="../figs-theory/es/E1c_explosion.svg" width="760"/>

**Siguiente (P2):** para razonar sobre todos estos caminos exactamente, necesitamos el álgebra de caminos `kQ`.

---

# P2 — El álgebra de caminos: contar caminos con álgebra lineal

El número de caminos entre dos nodos es exactamente lo que se aprieta. El **álgebra de caminos** `kQ` hace ese
número algebraico y computable: es igual a una entrada de una potencia de la matriz de adyacencia.

## 1. La matriz de adyacencia codifica las flechas

<img src="../figs-theory/es/E2a_adjacency_matrix.svg" width="760"/>

In [ ]:
# El quiver diamante y su matriz de adyacencia (aiq-quivers).
from aiq.quiver import Quiver
Q = Quiver([1,2,3,4], [('a1',1,2),('a2',1,3),('b1',2,4),('b2',3,4)])
A = Q.adjacency_matrix().astype(int)
print('A =\n', A)

## 2. A² cuenta los caminos de longitud 2

<img src="../figs-theory/es/E2b_A_squared.svg" width="760"/>

In [ ]:
# (A^2)[1,4] = 2 = los dos caminos 1->2->4 y 1->3->4. Verifícalo contra la enumeración.
import numpy as np
from aiq.path_algebra import PathAlgebra
A2 = A @ A
print('A^2 =\n', A2)
print('\n(A^2)[1,4] =', int(A2[0,3]))
print('paths_from_to(1,4,2):', [str(p) for p in PathAlgebra(Q).paths_from_to(1,4,2)])

## 3. Los caminos son la base de kQ — y la Proposición

La pieza graduada `e_j · kQ_g · e_i` tiene como base los caminos i→j de longitud g, así que su dimensión es `(A^g)[i,j]`.

<img src="../figs-theory/es/T3_paths_basis_kQ.svg" width="760"/>

In [ ]:
# dim(e_4 kQ_g e_1) = (A^g)[1,4] para todo g — la Proposición, comprobada numéricamente.
for g in range(1, 4):
    dim = len(PathAlgebra(Q).paths_from_to(1, 4, g))
    entry = int(np.linalg.matrix_power(A, g)[0, 3])
    print(f'g={g}:  dim(e4 kQ_{g} e1) = {dim}   (A^{g})[1,4] = {entry}   coincide: {dim==entry}')

## 4. La graduación: kQ = ⊕ kQ_g

La multiplicación es concatenación, y suma longitudes — así que el álgebra se separa limpiamente por longitud.

<img src="../figs-theory/es/T4_grading.svg" width="760"/>

**Siguiente (P3):** cada camino es un mensaje, así que `n_g(i,j)` es cuántos mensajes se acumulan.

---

# P3 — Los caminos son mensajes

Esto conecta P1 (over-squashing) y P2 (conteo de caminos). Tras `g` capas, un mensaje de `i` llega a `j`
**una vez por cada camino de longitud g**. Así que `n_g(i,j) = (A^g)[i,j]` es exactamente cuántas copias de la
información de `i` llegan a `j` — todas forzadas en un vector fijo.

## 1. Cada camino entrega un mensaje

<img src="../figs-theory/es/E3a_path_is_message.svg" width="760"/>

## 2. Los mensajes se apilan en un vector

Cuando la pila supera la capacidad del vector, el receptor no puede separarlos — over-squashing.

<img src="../figs-theory/es/E3b_messages_stack.svg" width="760"/>

In [ ]:
# Cuenta mensajes crudos vs el conteo de-duplicado (cociente) hacia el objetivo.
import torch
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash.ideal_bridge import walk_operators
K, M = 5, 4
for d in [1,2,3]:
    dd = make_bottleneck_graph(K, M, d, torch.Generator().manual_seed(0))
    raw, eff = walk_operators(dd.edge_index.numpy(), dd.num_nodes, max_length=d+1)
    t = int(dd.root_mask.nonzero()[0])
    print(f'profundidad {d}: crudo {int(raw[d][:,t].sum()):4d}   de-duplicado (kQ/I) {int(eff[d][:,t].sum()):3d}')

## 3. El cociente kQ/I: colapsar caminos equivalentes

Imponer una relación fusiona caminos paralelos en una clase, bajando la multiplicidad.

<img src="../figs-theory/es/T5_quotient.svg" width="760"/>

In [ ]:
# kQ/I colapsa los dos caminos de longitud 2 del diamante de 2 a 1 (vía aiq-quivers).
from oversquash.ideal_bridge import build_quotient_plan
import numpy as np
ei = np.array([[0,0,1,2],[1,2,3,3]])   # 0->1, 0->2, 1->3, 2->3
plan = build_quotient_plan(ei, num_nodes=4, max_length=2)
print('multiplicidad cruda (0->3, longitud 2):', plan.raw_mult.get((0,3,2)))
print('efectiva (kQ/I)                       :', plan.effective_mult.get((0,3,2)))

## 4. El giro: cuándo fusionar ayuda y cuándo daña

Colapsar caminos paralelos solo ayuda cuando son verdaderamente redundantes. En nuestra tarea de recuperación
la multiplicidad **es señal**, así que colapsarla quita información — un resultado negativo que confirmamos en
los experimentos.

<img src="../figs-theory/es/E3c_signal_vs_noise.svg" width="760"/>

**Siguiente (P4):** conservar todos los caminos, pero *aprender* cuánto pesar cada uno. Eso es atención.

---

# P4 — El operador de walk es atención

La idea central. Un Transformer atiende con pesos aprendidos sobre todos los pares; agregar sobre recorridos
de longitud g tiene la *misma forma*, pero el álgebra de caminos aporta un **soporte sparse, multi-salto**.
Walk Attention conserva los pesos aprendidos de la atención y toma el soporte del álgebra.

## 1. Atención de Transformer: una suma ponderada aprendida

<img src="../figs-theory/es/E4a_transformer_attention.svg" width="760"/>

## 2. El operador de walk tiene la misma forma

<img src="../figs-theory/es/T6_walk_is_attention.svg" width="760"/>

In [ ]:
# El soporte de atención en rango g es el patrón no-cero de A^g — sparse (vía aiq-quivers).
import torch, numpy as np
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash.ideal_bridge import walk_operators
data = make_bottleneck_graph(5, 4, 3, torch.Generator().manual_seed(0))
raw, _ = walk_operators(data.edge_index.numpy(), data.num_nodes, max_length=4)
N = data.num_nodes
for g in range(4):
    nz = int((raw[g] > 0).sum())
    print(f'rango g={g+1}: {nz:3d} pares alcanzables de {N*N}  ({100*nz/(N*N):.0f}% de todos los pares)')

## 3. El álgebra hace la atención sparse

La atención estándar llena todo el cuadrado; el soporte alcanzable por walks conserva solo ~2% en el rango más profundo.

<img src="../figs-theory/es/T7_dense_vs_sparse.svg" width="760"/>

## 4. Pesos fijos vs pesos aprendidos

El mismo soporte multi-salto — la única diferencia es pesos fijos (multiplicidad) vs aprendidos (query·key).
Esa diferencia es decisiva: los pesos aprendidos pueden *seleccionar* qué fuente importa.

<img src="../figs-theory/es/E4b_fixed_vs_learned.svg" width="760"/>

**Siguiente (P5):** entrena los tres modelos y mira cuál resuelve la tarea.

---

# P5 — Juntándolo todo

Tres modelos, una tarea de recuperación de largo alcance: **GAT** (1 salto, aprendido), el **operador de walk**
(multi-salto, pesos fijos) y **Walk Attention** (multi-salto, aprendido). La teoría predice el resultado; el
código lo confirma.

## 1. Tres operadores sobre el mismo grafo

<img src="../figs-theory/es/T8_three_operators.svg" width="760"/>

## 2. Dos ejes: soporte y pesos

<img src="../figs-theory/es/E5a_support_weights_table.svg" width="760"/>

## 3. Por qué gana o pierde cada modelo

<img src="../figs-theory/es/E5b_why_each.svg" width="760"/>

In [ ]:
# Entrena los tres modelos en la tarea de recuperación del cuello de botella (versión pequeña y rápida).
import torch, torch.nn.functional as F, numpy as np, math
from torch.utils.data import DataLoader
from torch_geometric.loader import DataLoader as PyGLoader
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash.transforms import (AttachWalkMasks, collate_walk_masks,
                                   AttachWalkOperators, collate_walk_operators)
from oversquash.models import build_model
K, M, DEPTH = 5, 4, 3   # azar = 1/5 = 0.20
def ds(seed, n=1500):
    g = torch.Generator().manual_seed(seed)
    return [make_bottleneck_graph(K, M, DEPTH, g) for _ in range(n)]

def train_eval(m, tr, va, fwd, epochs=70, lr=2e-3, warmup=8):
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda e: (e+1)/warmup if e<warmup
                                            else 0.5*(1+math.cos(math.pi*(e-warmup)/max(1,epochs-warmup))))
    best = 0.0
    for e in range(epochs):
        m.train()
        for b in tr:
            opt.zero_grad(); lg,_ = fwd(m,b)
            F.cross_entropy(lg, b.y, ignore_index=-100).backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        sch.step(); m.eval(); a=[]
        with torch.no_grad():
            for b in va:
                lg,_ = fwd(m,b); mm=b.root_mask
                a.append((lg[mm].argmax(-1)==b.y[mm]).float().mean().item())
        best = max(best, float(np.mean(a)))
    return best

def run(name, seed=0):
    torch.manual_seed(seed); data = ds(seed); nl = DEPTH+1
    if name=='gat':
        tr=PyGLoader(data[:1200],batch_size=128,shuffle=True); va=PyGLoader(data[1200:],batch_size=128)
        fwd=lambda m,b: m(b.x,b.edge_index,getattr(b,'batch',None))
    elif name=='walkraw':
        tf=AttachWalkOperators(n_layers=nl); data=[tf(d) for d in data]
        tr=DataLoader(data[:1200],batch_size=128,shuffle=True,collate_fn=collate_walk_operators)
        va=DataLoader(data[1200:],batch_size=128,collate_fn=collate_walk_operators)
        fwd=lambda m,b: m(b.x,b.edge_index,getattr(b,'batch',None),walk_raw=b.walk_raw)
    else:
        tf=AttachWalkMasks(n_layers=nl); data=[tf(d) for d in data]
        tr=DataLoader(data[:1200],batch_size=128,shuffle=True,collate_fn=collate_walk_masks)
        va=DataLoader(data[1200:],batch_size=128,collate_fn=collate_walk_masks)
        fwd=lambda m,b: m(b.x,b.edge_index,getattr(b,'batch',None),walk_masks=b.walk_masks)
    m = build_model(name, data[0].x.size(-1), 16, data[0].num_classes, nl, heads=4)
    return train_eval(m, tr, va, fwd)

for name in ['gat','walkraw','walkattn']:
    print(f'{name:10s} accuracy final = {run(name):.3f}')

## 4. El resultado

GAT cae al azar; el operador de walk alcanza pero no selecciona; Walk Attention lo resuelve (1.000 en 5 seeds
en la corrida completa).

<img src="../figs-theory/es/E5c_result.svg" width="760"/>

**El álgebra de caminos de un quiver define una atención sparse multi-salto que mitiga el over-squashing.**